# Azure OpenAI — Chat on Private Data (Production Setup)

This notebook implements a **persistent, production-grade RAG pipeline** using Azure OpenAI + LangChain + FAISS.

Key differences from a basic setup:
- ✅ FAISS index is **saved to disk** — survives restarts
- ✅ A `registry.json` tracks every indexed document (name, date, chunk count)
- ✅ Full **CRUD** — add, update, delete, list documents without re-indexing everything
- ✅ Multi-turn conversation history built in

---
### Folder structure created automatically
```
vector_store/
  faiss_index/      ← persisted FAISS binary files
  registry.json     ← tracks what documents are indexed
```

## 0 · Prerequisites

In [ ]:
!pip install langchain openai faiss-cpu python-dotenv tiktoken --quiet

Create a `.env` file in this folder with your Azure credentials:
```
OPENAI_API_KEY=xxxxxx
OPENAI_API_BASE=https://xxxxxxx.openai.azure.com/
```

---
## 1 · Configuration & Model Setup

In [ ]:
import os
import openai
from dotenv import load_dotenv
from langchain.chat_models import AzureChatOpenAI
from langchain.embeddings import OpenAIEmbeddings

load_dotenv()

openai.api_type    = "azure"
openai.api_version = "2023-03-15-preview"
openai.api_base    = os.getenv("OPENAI_API_BASE")
openai.api_key     = os.getenv("OPENAI_API_KEY")

llm = AzureChatOpenAI(
    deployment_name="gpt-4o",
    temperature=0,
    openai_api_version="2023-03-15-preview",
)
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002", chunk_size=1)

print("✅ Azure OpenAI configured")

---
## 2 · Vector Store Manager

This class handles everything related to storing, persisting, and managing your document vectors.

In [ ]:
import json
import logging
import shutil
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

from langchain.document_loaders import TextLoader
from langchain.text_splitter import TokenTextSplitter
from langchain.vectorstores import FAISS

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger(__name__)

# ── Paths ────────────────────────────────────────────────
STORE_ROOT    = Path("vector_store")
INDEX_PATH    = STORE_ROOT / "faiss_index"
REGISTRY_PATH = STORE_ROOT / "registry.json"
CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 100


# ── Registry helpers ─────────────────────────────────────
def _load_registry() -> dict:
    if REGISTRY_PATH.exists():
        with open(REGISTRY_PATH) as f:
            return json.load(f)
    return {}

def _save_registry(registry: dict) -> None:
    STORE_ROOT.mkdir(parents=True, exist_ok=True)
    with open(REGISTRY_PATH, "w") as f:
        json.dump(registry, f, indent=2)


# ── Manager class ─────────────────────────────────────────
class VectorStoreManager:
    """
    Persistent FAISS vector store with document-level CRUD.

    Methods
    -------
    add_document(path)      – index a new file
    add_directory(path)     – index all .txt files in a folder
    update_document(path)   – re-index a file after editing
    delete_document(path)   – remove a file's vectors
    list_documents()        – print what is currently indexed
    get_retriever()         – returns a LangChain retriever for QA chains
    """

    def __init__(self, embeddings):
        self.embeddings = embeddings
        self._db: Optional[FAISS] = None
        STORE_ROOT.mkdir(parents=True, exist_ok=True)
        self._load_or_init()

    def _load_or_init(self):
        if INDEX_PATH.exists():
            log.info("Loading existing FAISS index from %s", INDEX_PATH)
            self._db = FAISS.load_local(
                str(INDEX_PATH), self.embeddings,
                allow_dangerous_deserialization=True,
            )
            log.info("Index loaded. Documents tracked: %d", len(_load_registry()))
        else:
            log.info("No existing index — starting fresh.")
            self._db = None

    def _save(self):
        if self._db is not None:
            self._db.save_local(str(INDEX_PATH))

    @staticmethod
    def _load_and_split(file_path: str) -> list:
        loader = TextLoader(file_path, encoding="utf-8")
        docs   = loader.load()
        splitter = TokenTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
        chunks = splitter.split_documents(docs)
        for chunk in chunks:
            chunk.metadata["source"] = str(Path(file_path).resolve())
        return chunks

    def add_document(self, file_path: str):
        key = str(Path(file_path).resolve())
        registry = _load_registry()
        if key in registry:
            raise ValueError(f"'{file_path}' already indexed. Use update_document() instead.")
        chunks = self._load_and_split(file_path)
        if not chunks:
            log.warning("No content in %s — skipping.", file_path)
            return
        if self._db is None:
            self._db = FAISS.from_documents(chunks, self.embeddings)
        else:
            self._db.add_documents(chunks)
        registry[key] = {
            "filename":    Path(file_path).name,
            "indexed_at":  datetime.now(timezone.utc).isoformat(),
            "chunk_count": len(chunks),
        }
        _save_registry(registry)
        self._save()
        log.info("Added '%s' → %d chunks.", Path(file_path).name, len(chunks))

    def add_directory(self, dir_path: str, glob: str = "*.txt"):
        files = list(Path(dir_path).glob(glob))
        if not files:
            log.warning("No '%s' files found in %s", glob, dir_path)
            return
        for fp in files:
            try:
                self.add_document(str(fp))
            except ValueError as e:
                log.warning(str(e))

    def update_document(self, file_path: str):
        key = str(Path(file_path).resolve())
        registry = _load_registry()
        if key not in registry:
            log.info("'%s' not yet indexed — adding now.", file_path)
            self.add_document(file_path)
            return
        log.info("Updating '%s' — rebuilding index …", Path(file_path).name)
        self._rebuild_excluding(key)
        registry.pop(key, None)
        _save_registry(registry)
        self.add_document(file_path)

    def delete_document(self, file_path: str):
        key = str(Path(file_path).resolve())
        registry = _load_registry()
        if key not in registry:
            log.warning("'%s' not in registry — nothing to delete.", file_path)
            return
        self._rebuild_excluding(key)
        registry.pop(key)
        _save_registry(registry)
        self._save()
        log.info("Deleted '%s' from index.", Path(file_path).name)

    def list_documents(self):
        registry = _load_registry()
        if not registry:
            print("No documents indexed yet.")
            return
        print(f"\n{'─'*62}")
        print(f"  {'FILENAME':<32} {'CHUNKS':>6}  INDEXED AT (UTC)")
        print(f"{'─'*62}")
        for meta in registry.values():
            print(
                f"  {meta['filename']:<32} "
                f"{meta['chunk_count']:>6}  "
                f"{meta['indexed_at'][:19].replace('T', ' ')}"
            )
        print(f"{'─'*62}\n")

    def get_retriever(self, k: int = 4):
        if self._db is None:
            raise RuntimeError("No documents indexed yet. Call add_document() first.")
        return self._db.as_retriever(search_kwargs={"k": k})

    def _rebuild_excluding(self, exclude_source: str):
        if self._db is None:
            return
        all_docs  = list(self._db.docstore._dict.values())
        surviving = [d for d in all_docs if d.metadata.get("source") != exclude_source]
        if not surviving:
            self._db = None
            if INDEX_PATH.exists():
                shutil.rmtree(INDEX_PATH)
            return
        self._db = FAISS.from_documents(surviving, self.embeddings)
        self._save()


# Instantiate once; reuse throughout the notebook
store = VectorStoreManager(embeddings)
print("✅ VectorStoreManager ready")

---
## 3 · Ingest Documents

> **Run this section only once** (or when you add new files).  
> After the first run the index lives on disk — you don't need to re-ingest on every restart.

In [ ]:
# ── Option A: Ingest an entire folder ──────────────────────
store.add_directory("../data/qna/", glob="*.txt")

# ── Option B: Ingest a single file ─────────────────────────
# store.add_document("../data/qna/paris.txt")

---
## 4 · Inspect the Index

Run this any time — even after a restart — to see what's currently stored.

In [ ]:
store.list_documents()

---
## 5 · Build the QA Chain

In [ ]:
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts import PromptTemplate

CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template(
    """Given the following conversation and a follow-up question, rephrase \
the follow-up question to be a standalone question.

Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""
)

qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=store.get_retriever(k=4),
    condense_question_prompt=CONDENSE_QUESTION_PROMPT,
    return_source_documents=True,
    verbose=False,
)

print("✅ QA chain ready")

---
## 6 · Ask Questions (Multi-turn)

In [ ]:
def ask(question: str, chat_history: list) -> tuple[str, list]:
    """Ask a question and return (answer, updated_history)."""
    result  = qa({"question": question, "chat_history": chat_history})
    answer  = result["answer"]
    sources = {os.path.basename(d.metadata.get("source", "")) for d in result["source_documents"]}
    print(f"\n❓ {question}")
    print(f"💬 {answer}")
    if sources:
        print(f"📄 Sources: {', '.join(s for s in sources if s)}")
    return answer, chat_history + [(question, answer)]

In [ ]:
# Start a fresh conversation
chat_history = []

answer, chat_history = ask("Can you give me a brief of the AMAZING ANDAMAN package?", chat_history)

In [ ]:
# Follow-up — chat_history keeps context across cells
answer, chat_history = ask("What is included and excluded in the Paris package?", chat_history)

In [ ]:
answer, chat_history = ask("Give me a quick brief of the 6 Nights / 7 Days Swiss package.", chat_history)

---
## 7 · Document Management (CRUD)

These cells are your day-to-day operations — run them whenever you change your data.

In [ ]:
# ── ADD a new document ─────────────────────────────────────
# Uncomment and run when you add a new .txt file to your data folder

# store.add_document("../data/qna/new_package.txt")
# store.list_documents()

In [ ]:
# ── UPDATE a document (edited the file on disk) ────────────
# This deletes the old vectors and re-indexes the file cleanly

# store.update_document("../data/qna/paris.txt")
# store.list_documents()

In [ ]:
# ── DELETE a document ──────────────────────────────────────
# Removes all vectors for this document; other docs are untouched

# store.delete_document("../data/qna/old_package.txt")
# store.list_documents()

---
## 8 · Rebuild the QA Chain After Index Changes

If you added/updated/deleted documents, rebuild the retriever so the chain picks up the latest index.

In [ ]:
# Run this after any add / update / delete operation
qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=store.get_retriever(k=4),
    condense_question_prompt=CONDENSE_QUESTION_PROMPT,
    return_source_documents=True,
    verbose=False,
)
chat_history = []   # reset conversation too
print("✅ QA chain rebuilt with latest index")

---
## Quick Reference

| Task | Code |
|---|---|
| See what's indexed | `store.list_documents()` |
| Add a new file | `store.add_document("path/to/file.txt")` |
| Add a whole folder | `store.add_directory("path/to/folder/")` |
| Re-index edited file | `store.update_document("path/to/file.txt")` |
| Remove a file | `store.delete_document("path/to/file.txt")` |
| Ask a question | `ask("your question", chat_history)` |
| Reset conversation | `chat_history = []` |